# 🇹🇭 Thai NLP Toolkit — Training on Colab

Notebook นี้จะ:
1. Clone โปรเจคจาก GitHub
2. Upload data และ tokenizer model
3. ติดตั้ง dependencies
4. รัน training ด้วย GPU

> ⚠️ ก่อนเริ่ม: ไปที่ **Runtime → Change runtime type → GPU (T4)**

In [6]:
!git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 665 bytes | 665.00 KiB/s, done.
From https://github.com/puttibenz/thai-nlp-toolkit
   a3315c7..82050dd  main       -> origin/main
Updating a3315c7..82050dd
Fast-forward
 train_colab.ipynb | 18 ++++++++++++++++--
 1 file changed, 16 insertions(+), 2 deletions(-)


## 1. ตรวจสอบ GPU

In [7]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


## 2. Clone โปรเจคจาก GitHub

In [8]:
!git clone https://github.com/puttibenz/thai-nlp-toolkit.git
%cd thai-nlp-toolkit

Cloning into 'thai-nlp-toolkit'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 145 (delta 49), reused 120 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 76.44 KiB | 5.46 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/thai-nlp-toolkit/thai-nlp-toolkit


## 3. ติดตั้ง Dependencies

In [9]:
!pip install -q sentencepiece>=0.1.99 pythainlp>=4.0.0 datasets>=2.14.0 pyyaml>=6.0 tqdm>=4.65.0 scikit-learn>=1.3.0

## 4. Upload Data & Tokenizer Model

เนื่องจาก `data/raw/` และ `*.model` อยู่ใน `.gitignore`
ต้อง upload ขึ้นมาเอง

### วิธีที่ 1: Upload ผ่าน Google Drive (แนะนำ)
1. zip โฟลเดอร์ `data/raw/` และไฟล์ `tokenizer/thai_bpe.model`, `tokenizer/thai_bpe.vocab` รวมเป็น `thai_nlp_data.zip`
2. อัพขึ้น Google Drive
3. รันเซลล์ด้านล่าง

In [12]:
# === วิธีที่ 1: จาก Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# แก้ path ตามที่เก็บไฟล์ zip บน Drive
!cp /content/drive/MyDrive/thai_nlp_data.zip /content/thai-nlp-toolkit/
!unzip -o thai_nlp_data.zip
!rm thai_nlp_data.zip

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: cannot stat '/content/drive/MyDrive/thai_nlp_data.zip': No such file or directory
unzip:  cannot find or open thai_nlp_data.zip, thai_nlp_data.zip.zip or thai_nlp_data.zip.ZIP.
rm: cannot remove 'thai_nlp_data.zip': No such file or directory


In [16]:
from google.colab import files
uploaded = files.upload()  # เลือก thai_nlp_data.zip
!unzip -o thai_nlp_data.zip
!mkdir -p tokenizer/
!mv thai_bpe.model tokenizer/
!mv thai_bpe.vocab tokenizer/
!rm thai_nlp_data.zip

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_938/2664031138.py", line 2, in <cell line: 0>
    uploaded = files.upload()  # เลือก thai_nlp_data.zip
               ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/files.py", line 69, in upload
    uploaded_files = _upload_files(multiple=True)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/files.py", line 161, in _upload_files
    result = _output.eval_js(
             ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/output/_js.py", line 40, in eval_js
    return _message.read_reply_from_input(request_id, timeout_sec)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_messag

TypeError: object of type 'NoneType' has no len()

## 5. ตรวจสอบว่าไฟล์ครบ

In [15]:
import os

required_files = [
    "tokenizer/thai_bpe.model",
    "tokenizer/thai_bpe.vocab",
    "tokenizer/tokenizer_config.json",
    "data/raw/ner_train.jsonl",
    "data/raw/sent_train.tsv",
    "data/raw/qa_train.json",
]

for f in required_files:
    status = "✅" if os.path.exists(f) else "❌ MISSING"
    print(f"{status}  {f}")

❌ MISSING  tokenizer/thai_bpe.model
❌ MISSING  tokenizer/thai_bpe.vocab
✅  tokenizer/tokenizer_config.json
✅  data/raw/ner_train.jsonl
✅  data/raw/sent_train.tsv
✅  data/raw/qa_train.json


## 6. เริ่ม Training 🚀

In [ ]:
!python -m scripts.train --device cuda

## 7. ดาวน์โหลด Checkpoint กลับเครื่อง

In [ ]:
# บันทึกไป Google Drive
!cp -r outputs/ /content/drive/MyDrive/thai_nlp_outputs/
print("✅ saved to Google Drive")

# หรือดาวน์โหลดตรง
# !zip -r checkpoint.zip outputs/checkpoint_best/
# from google.colab import files
# files.download('checkpoint.zip')